# 05 Sentinel-1 SAR Flood Validation

This notebook checks whether the modelled flood-prone zones agree with observed historical flood evidence from Sentinel-1 SAR imagery.

## What you will do

1. Understand the SAR validation concept.
2. Check for pre-flood and flood/post-flood SAR rasters.
3. Confirm the modelled susceptibility class raster exists.
4. Run flood extent extraction and validation.
5. Read the validation metrics.

## Step 1: Validation concept

Sentinel-1 SAR can detect flooding because open water often causes a strong backscatter reduction. This starter workflow uses simple ratio/change detection between pre-flood and flood/post-flood rasters.

The model prediction is defined as:

- Class 4: High
- Class 5: Very High

These classes are compared against the SAR-derived observed flood extent.

In [ ]:
from pathlib import Path
import pandas as pd
from floodsense.config import load_config
from floodsense.paths import list_input_files

config = load_config()
paths = config['paths']

## Step 2: Check Sentinel-1 input files

Recommended placement:

- Pre-flood raster: `data/raw/sentinel1/pre_flood/`
- Flood/post-flood raster: `data/raw/sentinel1/during_flood/`

Use aligned GeoTIFF rasters where possible.

In [ ]:
sar_files = list_input_files(paths['raw_sentinel1'], ['.tif', '.tiff'])
print('Sentinel-1 raster count:', len(sar_files))
for file in sar_files:
    print(file)

## Step 3: Confirm modelled flood class raster

The validation step requires `susceptibility_class.tif` from notebook 03.

In [ ]:
modelled_class = Path(paths['processed_rasters']) / 'susceptibility_class.tif'
print('Modelled class raster:', modelled_class)
print('Exists:', modelled_class.exists())

## Step 4: Run SAR validation

Outputs:

- `data/processed/rasters/observed_flood_extent.tif`
- `data/processed/rasters/model_vs_observed_comparison.tif`
- `outputs/validation/validation_metrics.csv`

If Sentinel-1 rasters are missing, the workflow prints instructions and skips without creating fake results.

In [ ]:
from floodsense.sar_validation import run_sar_validation

run_sar_validation(config)

## Step 5: Read validation metrics

In [ ]:
metrics_path = Path(paths['outputs_validation']) / 'validation_metrics.csv'
if metrics_path.exists():
    validation_metrics = pd.read_csv(metrics_path)
    display(validation_metrics)
else:
    print('Validation metrics not available yet. Add Sentinel-1 rasters and rerun this notebook.')

## Step 6: Preview validation rasters

In [ ]:
from floodsense.mapping import plot_raster_preview

for name in ['observed_flood_extent', 'model_vs_observed_comparison']:
    raster = Path(paths['processed_rasters']) / f'{name}.tif'
    if raster.exists():
        plot_raster_preview(raster, Path(paths['outputs_maps']) / f'{name}_preview.png', name.replace('_', ' ').title())
        print('Preview saved:', name)
    else:
        print('Missing:', raster.name)

## What to check before moving on

- Precision: how many predicted flood pixels were actually observed as flood?
- Recall: how many observed flood pixels were captured by the model?
- F1 score: balance between precision and recall.
- Accuracy: overall agreement, but interpret carefully if flood pixels are rare.

Next notebook: `06_exposure_analysis.ipynb`.